In [ ]:
%pip install -q psycopg2-binary pymongo python-dotenv pandas

In [1]:
# Import the necessary libraries
import os
from urllib.parse import quote_plus

import pandas as pd
import psycopg2
from dotenv import load_dotenv
from pymongo import MongoClient
from psycopg2 import Error
from psycopg2.extras import execute_values

load_dotenv()

True

In [2]:
# PostgreSQL connection details from .env
hostname = os.getenv("hostnamePostgreSQL")
database = os.getenv("databasePostgreSQL")
port = os.getenv("portPostgreSQL")
username = os.getenv("usernamePostgreSQL")
password = os.getenv("passwordPostgreSQL")

connection = None
try:
    connection = psycopg2.connect(
        host=hostname,
        database=database,
        user=username,
        password=password,
        port=port,
    )
    cursor = connection.cursor()
    cursor.execute("SELECT version();")
    print("Connected to:", cursor.fetchone()[0])
except Error as error:
    print("Error while connecting to PostgreSQL:", error)
finally:
    if connection is not None:
        cursor.close()
        connection.close()
        print("PostgreSQL connection is closed")

Connected to: PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit
PostgreSQL connection is closed


In [3]:
# Preview the payment dataset
order_payments = pd.read_csv("data/olist_order_payments_dataset.csv")
order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [4]:
# Upload the payment dataset to PostgreSQL
csv_file_path = "data/olist_order_payments_dataset.csv"
table_name = "olist_order_payments"
connection = None

try:
    connection = psycopg2.connect(
        host=hostname,
        database=database,
        user=username,
        password=password,
        port=port,
    )
    cursor = connection.cursor()
    print("Connected to PostgreSQL successfully!")

    cursor.execute(f"DROP TABLE IF EXISTS {table_name};")
    cursor.execute(
        f"""
        CREATE TABLE {table_name} (
            order_id VARCHAR(50),
            payment_sequential INTEGER,
            payment_type VARCHAR(20),
            payment_installments INTEGER,
            payment_value NUMERIC(10, 2)
        );
        """
    )

    data = pd.read_csv(csv_file_path)
    batch_size = 500
    total_records = len(data)

    insert_query = f"""
        INSERT INTO {table_name}
        (order_id, payment_sequential, payment_type, payment_installments, payment_value)
        VALUES %s;
    """

    for start in range(0, total_records, batch_size):
        end = min(start + batch_size, total_records)
        batch_records = list(data.iloc[start:end].itertuples(index=False, name=None))
        execute_values(cursor, insert_query, batch_records)
        print(f"Inserted records {start + 1} to {end}")

    connection.commit()
    print(f"All {total_records} records inserted into {table_name}.")

except Error as error:
    if connection is not None:
        connection.rollback()
    print("Error while loading data into PostgreSQL:", error)
finally:
    if connection is not None:
        cursor.close()
        connection.close()
        print("PostgreSQL connection is closed")

Connected to PostgreSQL successfully!
Inserted records 1 to 500
Inserted records 501 to 1000
Inserted records 1001 to 1500
Inserted records 1501 to 2000
Inserted records 2001 to 2500
Inserted records 2501 to 3000
Inserted records 3001 to 3500
Inserted records 3501 to 4000
Inserted records 4001 to 4500
Inserted records 4501 to 5000
Inserted records 5001 to 5500
Inserted records 5501 to 6000
Inserted records 6001 to 6500
Inserted records 6501 to 7000
Inserted records 7001 to 7500
Inserted records 7501 to 8000
Inserted records 8001 to 8500
Inserted records 8501 to 9000
Inserted records 9001 to 9500
Inserted records 9501 to 10000
Inserted records 10001 to 10500
Inserted records 10501 to 11000
Inserted records 11001 to 11500
Inserted records 11501 to 12000
Inserted records 12001 to 12500
Inserted records 12501 to 13000
Inserted records 13001 to 13500
Inserted records 13501 to 14000
Inserted records 14001 to 14500
Inserted records 14501 to 15000
Inserted records 15001 to 15500
Inserted recor

In [ ]:
# MongoDB connection details from .env
mongo_hostname = os.getenv("hostnameMongoDB")
mongo_database = os.getenv("databaseMongoDB")
mongo_port = os.getenv("portMongoDB")
mongo_username = os.getenv("usernameMongoDB")
mongo_password = os.getenv("passwordMongoDB")

mongo_uri = (
    f"mongodb://{quote_plus(mongo_username)}:{quote_plus(mongo_password)}"
    f"@{mongo_hostname}:{mongo_port}/{mongo_database}"
)

client = MongoClient(mongo_uri)
mydatabase = client[mongo_database]
print("MongoDB client created")

In [ ]:
# Upload product-category translations to MongoDB
product_category_df = pd.read_csv(
    "data/product_category_name_translation.csv",
    encoding="utf-8-sig",
)

try:
    collection = mydatabase["product_categories"]
    records = product_category_df.to_dict(orient="records")
    result = collection.insert_many(records)
    print(f"Uploaded {len(result.inserted_ids)} records to MongoDB successfully!")
except Exception as error:
    print(f"An error occurred: {error}")
finally:
    client.close()
    print("MongoDB connection is closed")